In [22]:
import pandas as pd
import numpy as np

LFHV_RATES = {
    3:  {12: 700,  "13-17": 800,  "18-22": 900,  "23+": 1000},
    6:  {12: 900,  "13-17": 1000, "18-22": 1100, "23+": 1200},
    12: {12: 1100, "13-17": 1200, "18-22": 1400, "23+": 1600},
}

def _read_csv_safe(path):
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except:
        return pd.read_csv(path, encoding="latin1")

def _clean_money(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("₹", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

def _normalize_city(city):
    if pd.isna(city):
        return ""
    city = str(city).strip()
    if city.lower() == "baroda":
        return "Vadodara"
    return city

def _lfhv_slab(total):
    if total == 12:
        return 12
    elif 13 <= total <= 17:
        return "13-17"
    elif 18 <= total <= 22:
        return "18-22"
    elif total >= 23:
        return "23+"
    return None

def _hflv_rate(total):
    if 13 <= total <= 20:
        return 200
    elif 21 <= total <= 25:
        return 300
    elif 26 <= total <= 30:
        return 400
    elif total >= 31:
        return 500
    return 0

def _match_flat(city, source, speed, plan_amount):
    city = _normalize_city(city)
    source = str(source).strip().upper() if pd.notna(source) else ""
    speed = pd.to_numeric(pd.Series([speed]), errors="coerce").iloc[0]
    plan_amount = pd.to_numeric(pd.Series([plan_amount]), errors="coerce").iloc[0]

    if pd.isna(speed) or pd.isna(plan_amount):
        return None, "No flat"

    if city == "Vadodara" and speed == 20 and plan_amount == 2800:
        return 400, "Flat: Vadodara 20Mbps 2800"

    if city == "Surat" and source == "WB":
        if speed == 20 and plan_amount == 2998:
            return 400, "Flat: Surat WB 20Mbps 2998"
        elif speed == 30 and plan_amount == 3499:
            return 400, "Flat: Surat WB 30Mbps 3499"
        elif speed == 100 and plan_amount == 4949:
            return 400, "Flat: Surat WB 100Mbps 4949"

    if city in ["Surat", "Vadodara"] and speed == 20 and plan_amount in [1999, 3599]:
        return None, "In Slab"

    if city not in ["Surat", "Vadodara"]:
        if speed == 20 and plan_amount == 3599:
            return 400, "Flat: Other city 20Mbps 3599"
        elif speed == 20 and plan_amount == 1599:
            return 250, "Flat: Other city 20Mbps 1599"

    return None, "No flat"

def _parse_mixed_date(series):
    s = series.astype(str).str.strip()

    dt1 = pd.to_datetime(s, format="%d-%b-%y", errors="coerce")
    dt2 = pd.to_datetime(s, format="%d-%m-%y", errors="coerce")
    dt3 = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    dt4 = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")

    out = dt1.fillna(dt2).fillna(dt3).fillna(dt4)

    rem_mask = out.isna()
    if rem_mask.any():
        out.loc[rem_mask] = pd.to_datetime(s.loc[rem_mask], errors="coerce", dayfirst=True)

    return out

def calculate_payouts(sales_file, wb_file, ctc_file):
    # --- Load Data ---
    sales_df = _read_csv_safe(sales_file)
    wb_df = _read_csv_safe(wb_file)
    ctc_df = _read_csv_safe(ctc_file)

    # --- Clean columns ---
    sales_df.columns = sales_df.columns.str.strip()
    wb_df.columns = wb_df.columns.str.strip()
    ctc_df.columns = ctc_df.columns.str.strip()

    # --- Standardize EMP Code ---
    sales_df["EMP Code"] = sales_df["EMP Code"].astype(str).str.strip()
    wb_df["EMP Code"] = wb_df["EMP Code"].astype(str).str.strip()
    ctc_df["EMP Code"] = ctc_df["EMP Code"].astype(str).str.strip()

    # --- Filter CTC to Sales + Active ---
    ctc_df = ctc_df[
        (ctc_df["DEPARTMENT"].astype(str).str.strip().str.upper() == "SALES") &
        (ctc_df["SALES EXEC STATUS"].astype(str).str.strip().str.upper() == "ACTIVE")
    ].copy()

    ctc_df["Fixed_CTC"] = _clean_money(ctc_df["Fixed_CTC"]).fillna(0)
    ctc_df["SCHEME"] = ctc_df["SCHEME"].astype(str).str.strip().str.upper()

    ctc_map = ctc_df[["EMP Code", "Name", "Fixed_CTC", "SCHEME"]].drop_duplicates()

    # --- Filter Sales transactions if cols exist ---
    if "DEPARTMENT" in sales_df.columns:
        sales_df = sales_df[sales_df["DEPARTMENT"].astype(str).str.strip().str.upper() == "SALES"]
    if "SALES EXEC STATUS" in sales_df.columns:
        sales_df = sales_df[sales_df["SALES EXEC STATUS"].astype(str).str.strip().str.upper() == "ACTIVE"]

    # --- Filter WB transactions if cols exist ---
    if "DEPARTMENT" in wb_df.columns:
        wb_df = wb_df[wb_df["DEPARTMENT"].astype(str).str.strip().str.upper() == "SALES"]
    if "SALES EXEC STATUS" in wb_df.columns:
        wb_df = wb_df[wb_df["SALES EXEC STATUS"].astype(str).str.strip().str.upper() == "ACTIVE"]

    # --- Add source ---
    sales_df["Source"] = "New"
    wb_df["Source"] = "WB"

    # --- Combine ---
    combined = pd.concat([sales_df, wb_df], ignore_index=True)

    # --- Parse date ---
    combined["INSTALLATION DATE"] = _parse_mixed_date(combined["INSTALLATION DATE"])
    combined["MonthYear"] = combined["INSTALLATION DATE"].dt.strftime("%b-%Y")

    print("Total rows:", len(combined))
    print("Parsed dates:", combined["INSTALLATION DATE"].notna().sum())
    print("Unparsed dates:", combined["INSTALLATION DATE"].isna().sum())

    # --- Clean numeric and text columns ---
    combined["Plan Value"] = _clean_money(combined["Plan Value"])
    combined["SPEED (Mbps)"] = pd.to_numeric(combined["SPEED (Mbps)"], errors="coerce")
    combined["VALIDITY In Months"] = pd.to_numeric(combined["VALIDITY In Months"], errors="coerce")
    combined["City"] = combined["City"].apply(_normalize_city)

    # --- Merge CTC ---
    combined = combined.merge(ctc_map, on="EMP Code", how="inner")
    
    # --- Monthly context ---
    month_counts = (
        combined.groupby(["EMP Code", "MonthYear"], as_index=False)
        .agg(
            Installs=("Source", lambda x: (x == "New").sum()),
            Winbacks=("Source", lambda x: (x == "WB").sum()),
            Name=("Name", "first"),
            Fixed_CTC=("Fixed_CTC", "max"),
            SCHEME=("SCHEME", "first"),
            City=("City", "first")
        )
    )
    month_counts["Total Activations"] = month_counts["Installs"] + month_counts["Winbacks"]

    combined = combined.merge(
        month_counts[["EMP Code", "MonthYear", "Installs", "Winbacks", "Total Activations", "SCHEME"]],
        on=["EMP Code", "MonthYear"],
        how="left",
        suffixes=("", "_MONTH")
    )

    # --- Row-level payout logic ---
    def calc_row_payout(row):
        installs = row["Installs"]
        total = row["Total Activations"]
        scheme = str(row["SCHEME"]).strip().upper() if pd.notna(row["SCHEME"]) else ""
        validity = row["VALIDITY In Months"]
        city = row["City"]
        speed = row["SPEED (Mbps)"]
        plan_amount = row["Plan Value"]
        source = row["Source"]

        if installs < 8:
            return pd.Series([0.0, 0.0, "Not eligible: <8 installs"])

        slab_payout = 0.0
        remark = ""

        if scheme == "LFHV":
            if total < 12:
                return pd.Series([0.0, 0.0, "LFHV not eligible: <12 activations"])
            slab = _lfhv_slab(total)
            validity = int(validity) if pd.notna(validity) else None
            slab_payout = LFHV_RATES.get(validity, {}).get(slab, 0.0)
            remark = f"LFHV slab {slab} validity {validity}"

        elif scheme == "HFLV":
            if total < 13:
                return pd.Series([0.0, 0.0, "HFLV not eligible: <13 activations"])
            slab_payout = float(_hflv_rate(total))
            remark = f"HFLV rate {slab_payout}"

        else:
            return pd.Series([0.0, 0.0, "Unknown scheme"])

        flat_amt, flat_remark = _match_flat(city, source, speed, plan_amount)
        if flat_amt is not None:
            return pd.Series([float(flat_amt), 0.0, flat_remark])

        if pd.notna(plan_amount) and plan_amount < 2000:
            payable = float(slab_payout) * 0.5
            pending = float(slab_payout) * 0.5
            return pd.Series([payable, pending, remark + " | 50% payable low plan"])

        return pd.Series([float(slab_payout), 0.0, remark])

    combined[["ROW_PAYOUT", "ROW_PENDING", "ROW_REMARK"]] = combined.apply(calc_row_payout, axis=1)

    # --- Final monthly employee summary ---
    final = (
        combined.groupby(["EMP Code", "MonthYear"], as_index=False)
        .agg(
            Name=("Name", "first"),
            City=("City", "first"),
            Installs=("Source", lambda x: (x == "New").sum()),
            Winbacks=("Source", lambda x: (x == "WB").sum()),
            Total_Activations=("Source", "count"),
            Fixed_CTC=("Fixed_CTC", "max"),
            SCHEME=("SCHEME", "first"),
            Variable_Payout=("ROW_PAYOUT", "sum"),
            Pending_Payout=("ROW_PENDING", "sum")
        )
    )

    final["Total_Payout"] = final["Fixed_CTC"] + final["Variable_Payout"]
    final["CAC"] = final["Total_Payout"] / final["Total_Activations"].replace(0, np.nan)
    final["CAC"] = final["CAC"].fillna(0)

    num_cols = [
        "Installs", "Winbacks", "Total_Activations",
        "Fixed_CTC", "Variable_Payout", "Pending_Payout",
        "Total_Payout", "CAC"
    ]
    final[num_cols] = final[num_cols].round(0)

    final["MonthSort"] = pd.to_datetime(final["MonthYear"], format="%b-%Y", errors="coerce")
    final = final.sort_values(["MonthSort", "City", "Name"]).drop(columns=["MonthSort"]).reset_index(drop=True)

    return final, combined

In [23]:
monthly_cac, detail_rows = calculate_payouts(
    sales_file=r"C:\Users\Harshal Thakkar\Dashboard\Sales & Installation\New Registration Report.csv",
    wb_file=r"C:\Users\Harshal Thakkar\Dashboard\Sales & Installation\New Winback Report.csv",
    ctc_file=r"C:\Users\Harshal Thakkar\Dashboard\Sales & Installation\TSE-ACTIVE-CTC-31 Jul 2026.csv"
)

monthly_cac.head(20)

C:\Users\Harshal Thakkar\AppData\Local\Temp\ipykernel_18636\4229129977.py:98: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  out.loc[rem_mask] = pd.to_datetime(s.loc[rem_mask], errors="coerce", dayfirst=True)


Total rows: 13013
Parsed dates: 13013
Unparsed dates: 0


,EMP Code,MonthYear,Name,City,Installs,Winbacks,Total_Activations,Fixed_CTC,SCHEME,Variable_Payout,Pending_Payout,Total_Payout,CAC
0,10012300,Jul-2025,KISHAN MANJIBHAI SOLANKI,AHMEDABAD,8,5,13,17325,LFHV,11900.0,1300.0,29225.0,2248.0
1,10009453,Jul-2025,AMOL HANUMANTRAO NAIKNAWRE,AURANGABAD,10,3,13,20133,LFHV,6400.0,4400.0,26533.0,2041.0
2,10014393,Jul-2025,GORAKH MACHHINDRA KHAJEKAR,AURANGABAD,9,3,12,19051,LFHV,5300.0,3500.0,24351.0,2029.0
3,461597,Jul-2025,NASEEM AHMED SHAIKH,MUMBAI,7,0,7,35361,HFLV,0.0,0.0,35361.0,5052.0
4,10016295,Jul-2025,PRAVEEN LAKHANLAL SAROJ,MUMBAI,10,0,10,30100,HFLV,0.0,0.0,30100.0,3010.0
5,10016456,Jul-2025,RAJ KUMAR YADAV,MUMBAI,9,0,9,26100,HFLV,0.0,0.0,26100.0,2900.0
6,10012233,Jul-2025,AMAR JACHAK,NASHIK,15,7,22,19051,LFHV,22250.0,2350.0,41301.0,1877.0
7,10016413,Jul-2025,PAVAN DILIP AHIRE,NASHIK,4,0,4,23601,HFLV,0.0,0.0,23601.0,5900.0
8,10007861,Jul-2025,SAJEED SABIR SAYYAD,NASHIK,17,5,22,19051,LFHV,22850.0,2250.0,41901.0,1905.0
9,10007862,Jul-2025,SURAJ ASHOK KHODE,NASHIK,10,6,16,19051,LFHV,14200.0,2000.0,33251.0,2078.0


In [24]:
monthly_cac.shape
monthly_cac.columns.tolist()
monthly_cac.head()

,EMP Code,MonthYear,Name,City,Installs,Winbacks,Total_Activations,Fixed_CTC,SCHEME,Variable_Payout,Pending_Payout,Total_Payout,CAC
0,10012300,Jul-2025,KISHAN MANJIBHAI SOLANKI,AHMEDABAD,8,5,13,17325,LFHV,11900.0,1300.0,29225.0,2248.0
1,10009453,Jul-2025,AMOL HANUMANTRAO NAIKNAWRE,AURANGABAD,10,3,13,20133,LFHV,6400.0,4400.0,26533.0,2041.0
2,10014393,Jul-2025,GORAKH MACHHINDRA KHAJEKAR,AURANGABAD,9,3,12,19051,LFHV,5300.0,3500.0,24351.0,2029.0
3,461597,Jul-2025,NASEEM AHMED SHAIKH,MUMBAI,7,0,7,35361,HFLV,0.0,0.0,35361.0,5052.0
4,10016295,Jul-2025,PRAVEEN LAKHANLAL SAROJ,MUMBAI,10,0,10,30100,HFLV,0.0,0.0,30100.0,3010.0


In [25]:
detail_rows[
    [
        "EMP Code", "Name", "MonthYear", "Source", "City",
        "Plan Value", "SPEED (Mbps)", "VALIDITY In Months",
        "Installs", "Winbacks", "Total Activations",
        "SCHEME", "ROW_PAYOUT", "ROW_PENDING", "ROW_REMARK"
    ]
].head(30)

,EMP Code,Name,MonthYear,Source,City,Plan Value,SPEED (Mbps),VALIDITY In Months,Installs,Winbacks,Total Activations,SCHEME,ROW_PAYOUT,ROW_PENDING,ROW_REMARK
0,10015810,TUSHAR VIJAY KAJAVE,Jul-2025,New,PUNE,1500,60,3,12,2,14,LFHV,400.0,400.0,LFHV slab 13-17 validity 3 | 50% payable low plan
1,10013763,VANDANA KAMLESHBHAI RAJPUT,Jul-2025,New,VADODARA,3982,40,12,25,5,30,LFHV,1600.0,0.0,LFHV slab 23+ validity 12
2,10015833,GANESH RAMESH KANSE,Jul-2025,New,PUNE,1500,60,3,24,4,28,LFHV,500.0,500.0,LFHV slab 23+ validity 3 | 50% payable low plan
3,10014369,INAYAT ILYAS PATEL,Jul-2025,New,SURAT,2118,30,6,19,4,23,LFHV,1200.0,0.0,LFHV slab 23+ validity 6
4,10011924,JIMIT ASHOKBHAI MACHHI,Jul-2025,New,VADODARA,1500,60,3,25,0,25,LFHV,500.0,500.0,LFHV slab 23+ validity 3 | 50% payable low plan
5,10013775,MOHEMADNAZIF NAIMBHAI SHAIKH,Jul-2025,New,VADODARA,2118,40,6,25,9,34,LFHV,1200.0,0.0,LFHV slab 23+ validity 6
6,10015105,ANIL DEVAJI GANGARKAR,Jul-2025,New,PUNE,1500,60,3,18,5,23,LFHV,500.0,500.0,LFHV slab 23+ validity 3 | 50% payable low plan
7,10015554,SUSHANT SHANTARAM REMAJE,Jul-2025,New,PUNE,2286,125,3,17,3,20,LFHV,900.0,0.0,LFHV slab 18-22 validity 3
8,471416,DINESH KHUSHAL SHINDE,Jul-2025,New,VADODARA,3982,40,12,13,2,15,LFHV,1200.0,0.0,LFHV slab 13-17 validity 12
9,461597,NASEEM AHMED SHAIKH,Jul-2025,New,MUMBAI,1525,60,3,7,0,7,HFLV,0.0,0.0,Not eligible: <8 installs


In [26]:
monthly_cac[monthly_cac["Name"].isna()].head(20)

,EMP Code,MonthYear,Name,City,Installs,Winbacks,Total_Activations,Fixed_CTC,SCHEME,Variable_Payout,Pending_Payout,Total_Payout,CAC
